# BCH 디코더의 Berlekamp-Massey 알고리즘 학습 노트북

이 노트북은 BCH 코드 디코딩의 핵심인 **Berlekamp-Massey (BM) 알고리즘**을 단계별로 학습하기 위한 것입니다.

## 목차
1. [BCH 코드 개요](#1.-BCH-코드-개요)
2. [GF(2^m) 유한체 기초](#2.-GF(2^m)-유한체-기초)
3. [BCH 디코딩 3단계](#3.-BCH-디코딩-3단계)
4. [Berlekamp-Massey 알고리즘 이해](#4.-Berlekamp-Massey-알고리즘-이해)
5. [예시 1: 오류 1개 (t=1)](#5.-예시-1:-오류-1개)
6. [예시 2: 오류 2개 (t=2)](#6.-예시-2:-오류-2개)
7. [예시 3: 완전한 디코딩 과정](#7.-예시-3:-완전한-디코딩-과정)
8. [상세 분석 및 실험](#8.-상세-분석-및-실험)

In [ ]:
# 필요한 모듈 임포트
import sys
sys.path.append('..')

from bch_learning import (
    GaloisField, GFElement,
    BCHCode,
    BerlekampMassey,
    ChienSearch,
    gf_poly_str
)
from bch_learning.chien_search import decode_bch
import numpy as np

## 1. BCH 코드 개요

### BCH 코드란?

**BCH (Bose-Chaudhuri-Hocquenghem) 코드**는 강력한 오류 정정 능력을 가진 순환 부호입니다.

#### 파라미터
- **n**: 코드워드 길이
- **k**: 정보 비트 수
- **t**: 정정 가능한 최대 오류 개수
- **m**: 유한체 차수 (n = 2^m - 1)

#### 예시
- **BCH(15, 11, 1)**: 15비트 코드, 11비트 정보, 1비트 오류 정정
- **BCH(15, 7, 2)**: 15비트 코드, 7비트 정보, 2비트 오류 정정

### 왜 BCH 코드를 사용하는가?

1. **강력한 오류 정정**: 여러 비트 오류 정정 가능
2. **효율적인 디코딩**: 대수적 방법으로 빠른 디코딩
3. **유연성**: 원하는 오류 정정 능력에 맞게 설계 가능

## 2. GF(2^m) 유한체 기초

### GF(2)란?

**GF(2)**는 가장 간단한 유한체로, 원소는 {0, 1} 두 개뿐입니다.

- **덧셈**: XOR 연산
  - 0 + 0 = 0
  - 0 + 1 = 1
  - 1 + 0 = 1
  - 1 + 1 = 0

- **곱셈**: AND 연산
  - 0 × 0 = 0
  - 0 × 1 = 0
  - 1 × 0 = 0
  - 1 × 1 = 1

### GF(2^m)이란?

**GF(2^m)**은 2^m개의 원소를 가지는 유한체입니다.

- 각 원소는 m비트 이진수로 표현
- 원시 다항식(primitive polynomial)을 사용하여 구성
- 모든 0이 아닌 원소는 원시원소 α의 거듭제곱으로 표현

### GF(2^4) 예시

GF(2^4)는 16개의 원소를 가집니다: {0, 1, α, α^2, ..., α^14}

원시 다항식: p(x) = x^4 + x + 1

In [ ]:
# GF(2^4) 생성 및 테이블 출력
gf16 = GaloisField(4)
print(f"생성된 유한체: {gf16}")
print(f"원소 개수: {gf16.size}")
print(f"원시 다항식: {bin(gf16.primitive_poly)}")

# 유한체 테이블 출력
gf16.print_table()

### GF 연산 실습

In [ ]:
# GF(2^4) 원소 생성
a = gf16.alpha(3)  # α^3
b = gf16.alpha(5)  # α^5

print(f"a = {a}")
print(f"b = {b}")
print()

# 덧셈 (XOR)
c = a + b
print(f"a + b = {a} + {b} = {c}")
print(f"이진수: {a.to_binary()} XOR {b.to_binary()} = {c.to_binary()}")
print()

# 곱셈 (지수 덧셈)
d = a * b
print(f"a × b = {a} × {b} = {d}")
print(f"지수: 3 + 5 = 8 (mod 15)")
print()

# 나눗셈 (지수 뺄셈)
e = a / b
print(f"a / b = {a} / {b} = {e}")
print(f"지수: 3 - 5 = -2 = 13 (mod 15)")

## 3. BCH 디코딩 3단계

BCH 코드 디코딩은 다음 3단계로 구성됩니다:

### 1단계: 신드롬 계산 (Syndrome Computation)

수신된 코드워드 R(x)에 대해 신드롬을 계산합니다:

$$S_i = R(\alpha^i) \quad \text{for } i = 1, 2, \ldots, 2t$$

- 모든 신드롬이 0이면 오류 없음
- 신드롬이 0이 아니면 오류 존재

### 2단계: Berlekamp-Massey 알고리즘

신드롬으로부터 **오류 위치 다항식(Error Locator Polynomial, ELP)** Λ(x)를 찾습니다:

$$\Lambda(x) = 1 + \Lambda_1 x + \Lambda_2 x^2 + \cdots + \Lambda_L x^L$$

여기서 L은 오류 개수입니다.

**핵심 아이디어**: Λ(x)는 다음 신드롬 방정식을 만족합니다:

$$S_i + \Lambda_1 S_{i-1} + \Lambda_2 S_{i-2} + \cdots + \Lambda_L S_{i-L} = 0$$

### 3단계: Chien Search

Λ(x)의 근을 찾아 오류 위치를 결정합니다:

- Λ(α^(-i)) = 0 이면 위치 i에 오류 존재
- 모든 위치 i = 0, 1, ..., n-1을 검사

## 4. Berlekamp-Massey 알고리즘 이해

### 알고리즘의 목적

주어진 신드롬 시퀀스 S_1, S_2, ..., S_2t로부터 **최소 길이의 선형 피드백 시프트 레지스터(LFSR)**를 찾는 것입니다.

### 핵심 개념

#### 1. Discrepancy (불일치) Δ_k

현재 다항식 Λ^(k-1)(x)가 k번째 신드롬 S_k를 얼마나 잘 예측하는지 측정:

$$\Delta_k = S_k + \sum_{i=1}^{L} \Lambda_i S_{k-i}$$

- Δ_k = 0: 현재 다항식이 S_k를 완벽히 예측 → 다항식 유지
- Δ_k ≠ 0: 예측 실패 → 다항식 갱신 필요

#### 2. 다항식 갱신

Δ_k ≠ 0일 때, 다항식을 다음과 같이 갱신:

$$\Lambda^{(k)}(x) = \Lambda^{(k-1)}(x) + \frac{\Delta_k}{b} x^m B^{(k-1)}(x)$$

여기서:
- B(x): 이전에 저장된 다항식
- m: 마지막 갱신 이후 반복 횟수
- b: 마지막 갱신 시의 discrepancy

#### 3. 길이 갱신 조건

2L ≤ k이면 다항식 길이 L을 갱신:

$$L_{\text{new}} = k + 1 - L_{\text{old}}$$

### 알고리즘 의사코드

```
초기화:
  Λ(x) ← 1
  B(x) ← 1
  L ← 0
  m ← 1
  b ← 1

for k = 1 to 2t:
  # Discrepancy 계산
  Δ ← S_k + Σ(Λ_i × S_{k-i})
  
  if Δ = 0:
    m ← m + 1
  else:
    T ← Λ(x)
    Λ(x) ← Λ(x) + (Δ/b)·x^m·B(x)
    
    if 2L ≤ k:
      L ← k + 1 - L
      B(x) ← T
      b ← Δ
      m ← 1
    else:
      m ← m + 1

반환 Λ(x)
```

## 5. 예시 1: 오류 1개 (t=1)

### 설정

- BCH(15, 11, 1): 1비트 오류 정정
- 오류 위치: 위치 5
- 신드롬: S_1, S_2

### 수학적 배경

오류 위치가 i이면, 오류 위치 다항식은:

$$\Lambda(x) = 1 + \alpha^i x$$

신드롬과의 관계:

$$S_1 = \alpha^i$$
$$S_2 = \alpha^{2i}$$

In [ ]:
# BCH(15, 11, 1) 코드 생성
print("=" * 80)
print("예시 1: 오류 1개 케이스")
print("=" * 80)

gf = GaloisField(4)
bch = BCHCode(m=4, t=1, field=gf)

# 생성 다항식 출력
bch.print_generator_poly()

In [ ]:
# 메시지 인코딩
message = [1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1]  # 11비트 메시지
print(f"\n원본 메시지: {''.join(map(str, message))}")

codeword = bch.encode(message)
print(f"인코딩된 코드워드: {''.join(map(str, codeword))}")

# 오류 추가 (위치 5)
error_positions = [5]
received = bch.add_errors(codeword, error_positions)
print(f"\n오류 위치: {error_positions}")
print(f"수신된 코드워드: {''.join(map(str, received))}")
print(f"비교:            {''.join(map(str, codeword))}")
print(f"                 {''.join([' ' if c == r else '^' for c, r in zip(codeword, received)])}")

In [ ]:
# 신드롬 계산
syndromes = bch.compute_syndromes(received)
print(f"\n신드롬:")
for i, s in enumerate(syndromes, 1):
    print(f"  S_{i} = {s} (이진: {s.to_binary()})")

In [ ]:
# Berlekamp-Massey 알고리즘 실행
print("\n" + "=" * 80)
print("Berlekamp-Massey 알고리즘 실행")
print("=" * 80)

bm = BerlekampMassey(syndromes, verbose=True)
elp, iterations = bm.run()

# 요약 테이블
print(bm.get_summary_table())

In [ ]:
# 다항식 검증
bm.verify_polynomial()

In [ ]:
# Chien Search로 오류 위치 찾기
cs = ChienSearch(elp, bch.n, verbose=True)
found_errors = cs.search()

print(f"\n원래 오류 위치: {error_positions}")
print(f"찾은 오류 위치: {found_errors}")
print(f"일치 여부: {error_positions == found_errors}")

### 예시 1 분석

#### 신드롬 해석

오류가 위치 5에 있으므로:
- S_1 = α^5
- S_2 = α^10

#### BM 알고리즘 과정

**반복 1 (k=1)**: S_1 처리
- Δ_1 = S_1 = α^5 ≠ 0
- 2L = 0 ≤ 1이므로 길이 갱신
- Λ(x) = 1 + α^5·x

**반복 2 (k=2)**: S_2 처리
- Δ_2 = S_2 + α^5·S_1 = α^10 + α^5·α^5 = α^10 + α^10 = 0
- Δ = 0이므로 다항식 유지

**결과**:
- Λ(x) = 1 + α^5·x
- 근: α^(-5) = α^10 → 위치 5에 오류

## 6. 예시 2: 오류 2개 (t=2)

### 설정

- BCH(15, 7, 2): 2비트 오류 정정
- 오류 위치: 위치 3, 10
- 신드롬: S_1, S_2, S_3, S_4

### 수학적 배경

오류 위치가 i, j이면:

$$\Lambda(x) = (1 + \alpha^i x)(1 + \alpha^j x) = 1 + (\alpha^i + \alpha^j)x + \alpha^{i+j}x^2$$

신드롬:
$$S_k = \alpha^{ki} + \alpha^{kj}$$

In [ ]:
# BCH(15, 7, 2) 코드 생성
print("=" * 80)
print("예시 2: 오류 2개 케이스")
print("=" * 80)

bch2 = BCHCode(m=4, t=2, field=gf)
bch2.print_generator_poly()

In [ ]:
# 메시지 인코딩
message2 = [1, 0, 1, 1, 0, 1, 0]  # 7비트 메시지
print(f"\n원본 메시지: {''.join(map(str, message2))}")

codeword2 = bch2.encode(message2)
print(f"인코딩된 코드워드: {''.join(map(str, codeword2))}")

# 오류 추가 (위치 3, 10)
error_positions2 = [3, 10]
received2 = bch2.add_errors(codeword2, error_positions2)
print(f"\n오류 위치: {error_positions2}")
print(f"수신된 코드워드: {''.join(map(str, received2))}")
print(f"비교:            {''.join(map(str, codeword2))}")
print(f"                 {''.join([' ' if c == r else '^' for c, r in zip(codeword2, received2)])}")

In [ ]:
# 신드롬 계산
syndromes2 = bch2.compute_syndromes(received2)
print(f"\n신드롬:")
for i, s in enumerate(syndromes2, 1):
    print(f"  S_{i} = {s}")

In [ ]:
# Berlekamp-Massey 알고리즘 실행 (상세)
print("\n" + "=" * 80)
print("Berlekamp-Massey 알고리즘 실행 (오류 2개)")
print("=" * 80)

bm2 = BerlekampMassey(syndromes2, verbose=True)
elp2, iterations2 = bm2.run()

# 요약 테이블
print(bm2.get_summary_table())

In [ ]:
# 각 반복 단계 상세 분석
print("\n" + "=" * 80)
print("각 반복 단계 상세 분석")
print("=" * 80)

for it in iterations2:
    print(f"\n[반복 {it.iteration}]")
    print(f"  입력 신드롬: S_{it.iteration} = {it.syndrome_used}")
    print(f"  Discrepancy: Δ = {it.discrepancy}")
    print(f"  현재 다항식: Λ(x) = {gf_poly_str(it.current_poly)}")
    print(f"  다항식 길이: L = {it.length}")
    print(f"  m 값: {it.m_value}")
    print(f"  갱신 여부: {'예' if it.update_occurred else '아니오'}")
    print(f"  설명: {it.explanation}")

In [ ]:
# 다항식 검증
bm2.verify_polynomial()

In [ ]:
# Chien Search
cs2 = ChienSearch(elp2, bch2.n, verbose=True)
found_errors2 = cs2.search()

print(f"\n원래 오류 위치: {error_positions2}")
print(f"찾은 오류 위치: {found_errors2}")
print(f"일치 여부: {set(error_positions2) == set(found_errors2)}")

### 예시 2 분석

#### 오류 2개의 수학적 관계

오류 위치를 i, j라 하면:

$$\Lambda(x) = (1 + \alpha^i x)(1 + \alpha^j x)$$
$$= 1 + (\alpha^i + \alpha^j)x + \alpha^{i+j}x^2$$

따라서:
- Λ_1 = α^i + α^j
- Λ_2 = α^(i+j)

#### 신드롬 방정식

- S_1 = α^i + α^j
- S_2 = α^(2i) + α^(2j)
- S_3 = α^(3i) + α^(3j)
- S_4 = α^(4i) + α^(4j)

#### BM 알고리즘의 4번 반복

각 반복에서 하나씩 신드롬을 처리하며, discrepancy가 0이 아니면 다항식을 갱신합니다.

최종적으로 2차 다항식 Λ(x)를 얻고, 이것의 두 근이 오류 위치를 나타냅니다.

## 7. 예시 3: 완전한 디코딩 과정

이제 전체 디코딩 과정을 하나의 흐름으로 실행해봅시다.

In [ ]:
print("=" * 80)
print("완전한 BCH 디코딩 예시")
print("=" * 80)

# 1. 코드 생성
bch_full = BCHCode(m=4, t=2, field=gf)
print(f"\nBCH({bch_full.n}, {bch_full.k}, {bch_full.t}) 코드 사용")

# 2. 메시지 인코딩
msg = [1, 1, 0, 1, 0, 1, 1]
print(f"원본 메시지: {''.join(map(str, msg))}")

code = bch_full.encode(msg)
print(f"인코딩: {''.join(map(str, code))}")

# 3. 오류 추가
errors = [2, 8]
recv = bch_full.add_errors(code, errors)
print(f"\n오류 추가 (위치 {errors})")
print(f"수신: {''.join(map(str, recv))}")
print(f"원본: {''.join(map(str, code))}")
print(f"      {''.join([' ' if c == r else '^' for c, r in zip(code, recv)])}")

In [ ]:
# 4. 완전한 디코딩 실행
corrected, success = decode_bch(recv, bch_full, verbose=True)

# 5. 결과 확인
print("\n" + "=" * 80)
print("디코딩 결과 요약")
print("=" * 80)
print(f"수신: {''.join(map(str, recv))}")
print(f"정정: {''.join(map(str, corrected))}")
print(f"원본: {''.join(map(str, code))}")
print(f"\n정정 성공: {success}")
print(f"일치 여부: {corrected == code}")

## 8. 상세 분석 및 실험

### 실험 1: 다양한 오류 패턴 테스트

In [ ]:
def test_error_pattern(bch_code, error_positions, test_name):
    """특정 오류 패턴 테스트"""
    print(f"\n{'='*60}")
    print(f"테스트: {test_name}")
    print(f"오류 위치: {error_positions}")
    print(f"{'='*60}")
    
    # 임의 메시지
    msg = [np.random.randint(0, 2) for _ in range(bch_code.k)]
    code = bch_code.encode(msg)
    recv = bch_code.add_errors(code, error_positions)
    
    # 디코딩
    corrected, success = decode_bch(recv, bch_code, verbose=False)
    
    print(f"\n결과: {'성공' if success else '실패'}")
    if success:
        print(f"원본과 일치: {corrected == code}")
    
    return success

# 다양한 오류 패턴 테스트
bch_test = BCHCode(m=4, t=2, field=gf)

test_patterns = [
    ([0], "단일 오류 (위치 0)"),
    ([7], "단일 오류 (위치 7)"),
    ([0, 14], "양 끝 오류"),
    ([5, 6], "인접 오류"),
    ([1, 8], "분산 오류"),
]

results = []
for errors, name in test_patterns:
    success = test_error_pattern(bch_test, errors, name)
    results.append(success)

print(f"\n{'='*60}")
print(f"전체 테스트 결과: {sum(results)}/{len(results)} 성공")
print(f"{'='*60}")

### 실험 2: BM 알고리즘 수렴 관찰

In [ ]:
import matplotlib.pyplot as plt

def plot_bm_convergence(iterations):
    """BM 알고리즘의 수렴 과정 시각화"""
    iters = [it.iteration for it in iterations]
    lengths = [it.length for it in iterations]
    disc_nonzero = [0 if it.discrepancy.is_zero() else 1 for it in iterations]
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    
    # 다항식 길이 변화
    ax1.plot(iters, lengths, 'o-', linewidth=2, markersize=8)
    ax1.set_xlabel('반복 횟수', fontsize=12)
    ax1.set_ylabel('다항식 길이 L', fontsize=12)
    ax1.set_title('Berlekamp-Massey: 다항식 길이 변화', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(iters)
    
    # Discrepancy 0/비0
    ax2.bar(iters, disc_nonzero, color=['green' if d == 0 else 'red' for d in disc_nonzero])
    ax2.set_xlabel('반복 횟수', fontsize=12)
    ax2.set_ylabel('Discrepancy 상태', fontsize=12)
    ax2.set_title('Discrepancy (빨강=비0, 초록=0)', fontsize=14, fontweight='bold')
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['Δ=0', 'Δ≠0'])
    ax2.set_xticks(iters)
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

# 예시 2의 결과로 시각화
plot_bm_convergence(iterations2)

### 실험 3: GF(2^m) 크기 비교

In [ ]:
# 다양한 m 값에 대한 BCH 코드 파라미터
print("\nGF(2^m) 크기에 따른 BCH 코드 파라미터\n")
print(f"{'m':<4} {'n':<6} {'t':<4} {'k':<6} {'코드율':<10}")
print("-" * 35)

for m in range(3, 8):
    for t in [1, 2]:
        try:
            field_m = GaloisField(m)
            bch_m = BCHCode(m=m, t=t, field=field_m)
            rate = bch_m.k / bch_m.n
            print(f"{m:<4} {bch_m.n:<6} {t:<4} {bch_m.k:<6} {rate:<10.3f}")
        except:
            pass

## 9. 핵심 개념 정리

### Berlekamp-Massey 알고리즘의 핵심

1. **목적**: 신드롬 시퀀스를 생성하는 최소 길이의 LFSR 찾기

2. **Discrepancy**: 현재 다항식의 예측 정확도 측정
   - Δ = 0: 완벽한 예측
   - Δ ≠ 0: 다항식 갱신 필요

3. **길이 갱신**: 2L ≤ k 조건
   - 만족: 다항식 길이 증가 (더 복잡한 패턴)
   - 불만족: 길이 유지 (미세 조정)

4. **효율성**: O(n^2) 시간 복잡도 (n: 신드롬 개수)

### BCH 디코딩 흐름

```
수신 코드워드 R(x)
    ↓
[1단계] 신드롬 계산
    S_i = R(α^i)
    ↓
[2단계] Berlekamp-Massey
    신드롬 → 오류 위치 다항식 Λ(x)
    ↓
[3단계] Chien Search
    Λ(x)의 근 찾기 → 오류 위치
    ↓
오류 정정
```

### 수학적 아름다움

- **대수적 구조**: 유한체의 아름다운 성질 활용
- **효율성**: 복잡한 문제를 다항식 연산으로 단순화
- **일반성**: 다양한 오류 패턴에 대응 가능

## 10. 추가 실험

여기서 자유롭게 실험해보세요!

In [ ]:
# 여기에 자유롭게 코드를 작성하여 실험해보세요
# 예시:
# - 다른 오류 패턴 테스트
# - 다른 BCH 파라미터 시도
# - GF 연산 실습
# - BM 알고리즘 상세 분석

# 코드 작성 공간


## 참고 자료

1. **교재**
   - "Error Control Coding" by Shu Lin and Daniel J. Costello
   - "Algebraic Codes for Data Transmission" by Richard E. Blahut

2. **논문**
   - Berlekamp, E. R. (1968). "Algebraic Coding Theory"
   - Massey, J. L. (1969). "Shift-register synthesis and BCH decoding"

3. **온라인 리소스**
   - Wikipedia: BCH code, Berlekamp-Massey algorithm
   - MIT OpenCourseWare: Coding Theory